# 09주 · 벡터·행렬·질량보존·고유값

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 원리·도구 기능 이해 → 좋은 AI 질문·지시 → 오차·수렴·결과의 검토

**수업**: 2027-04-27 선형 연립방정식 (1) · 대면 / 2027-04-29 선형 연립방정식 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week09/CE273_W09.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week09)

슬라이드 예측 → 작은 손계산 → 실행 → 한 조건 변경 → AI 요청 수정 → 단위·오차·수렴·실패 상태 검토

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def linear_system(A, b):
    A=np.asarray(A,dtype=float);b=vector(b)
    if A.ndim!=2 or A.shape[0]!=A.shape[1] or A.shape[0]!=len(b) or not np.isfinite(A).all():
        raise ValueError("정방행렬·차원·유한한 값 확인 필요")
    x=np.linalg.solve(A,b)
    residual=A@x-b
    den=np.linalg.norm(A,np.inf)*np.linalg.norm(x,np.inf)+np.linalg.norm(b,np.inf)
    return {"x":x,"residual":residual,"backward_error":0. if den==0 else np.linalg.norm(residual,np.inf)/den,"condition_2":np.linalg.cond(A)}

def gaussian_pivot(A,b):
    A=np.asarray(A,dtype=float).copy();b=vector(b).copy()
    if A.ndim!=2 or A.shape!=(len(b),len(b)) or not np.isfinite(A).all():raise ValueError("정방행렬·우변 차원 확인 필요")
    n=len(b)
    for k in range(n):
        pivot=k+np.argmax(abs(A[k:,k]))
        if abs(A[pivot,k])<=np.finfo(float).eps*np.linalg.norm(A,np.inf):raise np.linalg.LinAlgError("0 또는 작은 피벗: 특이성·스케일 검토")
        A[[k,pivot]]=A[[pivot,k]];b[[k,pivot]]=b[[pivot,k]]
        for i in range(k+1,n):
            factor=A[i,k]/A[k,k];A[i,k:]-=factor*A[k,k:];b[i]-=factor*b[k]
    x=np.zeros(n)
    for i in range(n-1,-1,-1):x[i]=(b[i]-A[i,i+1:]@x[i+1:])/A[i,i]
    return x


## S01. 원본 벡터와 반응조 선형계

**슬라이드 연결**: 2026_S1_NA_Wk9a_LinearAlgebra1 .pptx p5, 7, 14, 15 / 2026_S1_NA_Wk9b_LinearAlgebra2.pptx p3, 4, 8

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 행렬 한 행은 어떤 보존식을 뜻하는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
u=np.array([1.,2,3]);v=np.array([4.,5,6])
print("dot:",u@v,"cross:",np.cross(u,v));assert u@v==32
A=np.array([[6,0,-1,0,0],[-3,3,0,0,0],[0,-1,9,0,0],[0,-1,-8,11,-2],[-3,-1,0,0,4]],dtype=float)
b=np.array([50,0,160,0,0],dtype=float);out=linear_system(A,b)
print("반응조 농도(mg/m³):",out["x"],"backward_error:",out["backward_error"])
assert np.all(out["x"]>0) and np.linalg.norm(out["residual"])<1e-10
print("*는 원소별, @는 행렬곱")

**조건 변경**: 우변 유입량 하나만 변경 후 농도·잔차 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 계수행렬·우변·변수 순서·유량/농도 단위·잔차 검산 명시

**결과 검토**: 역행렬을 직접 구하는 대신 solve로 계산, 원래 보존식에 대입

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. 원본 대칭행렬의 고유쌍

**슬라이드 연결**: 2026_S1_NA_Wk9a_LinearAlgebra1 .pptx p25, 26, 27, 28 / 2026_S1_NA_Wk9b_LinearAlgebra2.pptx p11, 15, 16, 17, 19

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 고유벡터의 부호가 원본과 다르면 오류인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
A=np.array([[2.,1],[1,2]])
values,V=np.linalg.eigh(A)
print("eigenvalues:",values,"eigenvectors(columns):",V)
assert np.allclose(values,[1,3]) and np.allclose(A@V,V*values)
assert np.allclose(A,V@np.diag(values)@V.T)
print("고유벡터의 부호·배율과 고유쌍의 나열 순서는 유일하지 않음")

**조건 변경**: 같은 벡터를 -1배 한 사본에서 고유식 검산

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 대칭 여부·고유벡터 열·고유값 순서·잔차·재구성 확인 요청

**결과 검토**: 모든 행렬의 대각화 가능성이나 직교 고유벡터를 가정하지 않음

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

행렬곱·해·고유쌍의 정의와 잔차 설명

입력 항목:

- `벡터_내적`: 해당 계산값·항 수·오차 기준의 수치
- `최소_고유값`: 해당 계산값·항 수·오차 기준의 수치
- `최대_고유값`: 해당 계산값·항 수·오차 기준의 수치
- `고유벡터_부호가_다르면_항상_오류`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'벡터_내적': None, '최소_고유값': None, '최대_고유값': None, '고유벡터_부호가_다르면_항상_오류': None}


In [ ]:
def check_answers(values):
    expected = {'벡터_내적': 32.0, '최소_고유값': 1.0, '최대_고유값': 3.0, '고유벡터_부호가_다르면_항상_오류': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

1. 문제·모형·입력·단위·자료 출처
2. 최초 요청: 목적·조건·알고리즘·출력·검증 기준
3. 실행 전 예측·작은 손계산의 근거
4. 의심한 출력·오차·수렴·실패 조건과 수정 지시
5. 채택·수정·거절한 내용, 독립 검산·한계
6. AI 없이 설명 가능한 원리 한 가지

글자 수 자동 채점 없음. 실행 성공·작은 갱신량·작은 잔차와 문제 해결의 타당성 구분

In [ ]:
review_log = {
    "문제·모형·자료·단위": None,
    "최초 AI 요청": None,
    "예측·손계산": None,
    "의심한 부분·수정 지시": None,
    "검산·최종 판단·한계": None,
    "내 설명": None,
}
